# OLMo-2-1B × Tulu-3 SFT mixture leaderboard — robustness pilot (9000 steps, ~225M slot-tokens)

Robustness pilot from `~/.claude/plans/as-part-of-our-tender-quilt.md`. Tests whether `adam-polar-product-lora-coupled-spectral-chord-tight` (plain k=1) keeps its eval-loss advantage over AdamW-LoRA when the dataset is swapped from opc-sft-stage2 (code-IFT) to Tulu-3 (general-IFT, chat-templated via the open-instruct Tulu chat template).

Cell: OLMo-2-1B × Tulu-3 SFT mixture (400k subset, ~150k packed slots @ seq=2048) × global_batch=16 (batch=4 × accum=4) × packed_v1.1 × constant LR × α=r × all-linear × bf16 × compile × single-GPU Blackwell. `max_steps=9000`, `eval_every=250`.

- **AdamW**: η ∈ {3e-5, 1e-4, 3e-4}
- **chord-tight k=1** (`adam-polar-product-lora-coupled-spectral-chord-tight`): η ∈ {3e-3, 1e-2, 3e-2}

Source log groups: `{adamw,chord_tight}_robustness_tulu3_1b_r{64,256}_blackwell` (4 groups total).

**σ anchor**: no per-dataset multi-seed AdamW run yet. Quoting Δ against `σ_AdamW(packed_v1, opc-sft-stage2, r=64) = 0.0017` as a **proxy only** — re-anchor before any paper claim.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lora_playground.loader import load_runs
from lora_playground.plotting import compare_variants_figure, canonical_label
from IPython.display import display

# Single source of truth: canonical_label (ns/picard/damping-explicit, identical
# across every notebook), canonical colors (AdamW black + first), guard hard-errors
# on any silent merge.

def _key(cfg):
    return (cfg['optimizer'], float(cfg['lr']), cfg.get('muon_ns_steps'),
            cfg.get('_derived', {}).get('effective_picard_iters', cfg.get('picard_iters_override')))

def render_cell(groups, rank, suptitle, *, figsize=(11, 4)):
    runs = load_runs(where={'log_group': groups}, logs_root='../logs',
                     warn_cross_commit=False, quiet=True)
    dedup = {}
    for cfg, hist in runs:
        if cfg.get('lora_r') != rank:
            continue
        k = _key(cfg)
        if k not in dedup or len(hist) > len(dedup[k][1]):
            dedup[k] = (cfg, hist)
    labeled = [(c, h) for c, h in dedup.values() if canonical_label(c) is not None]
    labels = {canonical_label(c) for c, _ in labeled}
    # Achieved horizon, NOT a hardcoded 9000: tulu3 is smaller than 9000 steps'
    # worth, so runs exhaust at one epoch (~8970). The final-vs-lr panel gates on
    # last_step == max_steps, so passing 9000 would classify every run "partial"
    # and drop it. Use the longest completed run's step as the horizon.
    _last = [h[-1]['step'] for _, h in labeled if h]
    horizon = max(_last) if _last else 9000
    fig, tdf, sdf = compare_variants_figure(
        variants={l: {} for l in labels}, common_where={}, ref_label='AdamW',
        target_label='AdamW', sigma_ref=0.0017, suptitle=suptitle, figsize=figsize,
        max_steps=horizon, allow_partial=True,
        prefetched_runs=labeled, variant_key=canonical_label)
    plt.show()
    print(f'--- {suptitle} per-η table ---')
    display(tdf.style.format('{:.4f}', na_rep='—'))
    print(f'--- {suptitle} summary ---')
    display(sdf.style.format({'final': '{:.4f}', 'delta': '{:+.4f}',
                              'delta_sigma': '{:+.2f}σ', 'best_lr': '{:.0e}'}, na_rep='—'))
    return tdf, sdf

## r=64

In [ ]:
GROUPS_R64 = [
    'adamw_robustness_tulu3_1b_r64_blackwell',
    'adamw_robustness_tulu3_1b_r64_gpuxl',
    'chord_tight_robustness_tulu3_1b_r64_gpuxl',
]
render_cell(GROUPS_R64, 64, 'Tulu-3 r=64')

## r=256

In [ ]:
GROUPS_R256 = [
    'adamw_robustness_tulu3_1b_r256_blackwell',
    'adamw_robustness_tulu3_1b_r256_gpuxl',
    'chord_tight_robustness_tulu3_1b_r256_gpuxl',
    'chord_tight_robustness_tulu3_1b_r256_ext_right_gpuxl',  # right-extension (chord r256 pinned at 3e-2)
]
render_cell(GROUPS_R256, 256, 'Tulu-3 r=256')